In [1]:
from mdt.datasets.tcl_dataset import TCLImageDataset
import mediapy
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from robokit.debug_utils.images import concatenate_rgb_images, plot_action_wrt_time
from robokit.debug_utils.io import dataloader_speed_test

In [4]:
dataset = TCLImageDataset(
    data_root="/home/geyuan/local_soft/TCL/0709_coffee_source",
    h5_path="/home/geyuan/local_soft/TCL/hdf5/0709_coffee_source_240p.h5",
    use_h5=True,
    horizon=16, pad_before=0, pad_after=7,
    shape_meta={
        "obs": {
            "image": {
                "shape": [3, 224, 224],
                "type": "rgb"
            },
            "gripper": {
                "shape": [3, 224, 224],
                "type": "rgb"
            },
            "joint_state": {
                "shape": [6],
                "type": "low_dim"
            }
        },
        "action": {
            "shape": [7,]
        }
    },
    norm_action_type="mean",
    transform_color_jitter=True,
)

[TCLDataset] loaded key=rel_actions shape=(117009, 7) from /home/geyuan/local_soft/TCL/0709_coffee_source/extracted/rel_actions.npy
[TCLDataset] total length: 117009
[TCLDatasetHDF5] using h5 data: /home/geyuan/local_soft/TCL/hdf5/0709_coffee_source_240p.h5
[TCLDataset] loading dataset statistics from: /home/geyuan/local_soft/TCL/0709_coffee_source/statistics.json
[TCLImageDataset] dataset loaded, split=train, val_ratio=0.01, len=115839; total_len=117009, norm_type=mean, action_min=[-0.09999695 -0.08509827 -0.09529114 -0.77414072 -0.5193474  -0.67749035
  0.        ], action_max=[0.09607849 0.09450684 0.1        0.51976277 0.77972236 0.45509258
 1.        ], action_mean=[-4.19806647e-03  4.17534582e-04  5.40954816e-04 -1.35875802e-03
 -8.98782482e-03  2.93930787e-03  8.11971729e-01], action_std=[0.0334569  0.02095223 0.03304508 0.08959766 0.07085136 0.04801901
 0.39073474]


In [5]:
dataloader_speed_test(dataset, num_workers=48)

workers=48, batch=64:   0%|                                                                                                | 1/1810 [00:12<6:25:59, 12.80s/it]

batch_data@0: Dict,keys=dict_keys(['robot_obs', 'rgb_obs', 'depth_obs', 'actions', 'state_info', 'lang', 'lang_text', 'idx', 'future_frame_diff'])
robot_obs,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 8])
rgb_obs: Dict,keys=dict_keys(['rgb_static', 'rgb_gripper', 'gen_static', 'gen_gripper'])
-rgb_static,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 3, 224, 224])
-rgb_gripper,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 3, 224, 224])
-gen_static,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 3, 112, 112])
-gen_gripper,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 3, 112, 112])
depth_obs: Dict,keys=dict_keys([])
actions,<class 'torch.Tensor'>,shape=torch.Size([64, 16, 7])
state_info: Dict,keys=dict_keys(['scene_obs', 'robot_obs'])
-scene_obs,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 24])
-robot_obs,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 15])
lang: Dict,keys=dict_keys([])
lang_text: List,len=64,elem:<class 'str'>
idx,<class 'torch.Tensor'>,shape=torch.Size([64]

workers=48, batch=64:   8%|███████▋                                                                                        | 146/1810 [00:25<04:45,  5.84it/s]
Exception in thread Thread-6:
Traceback (most recent call last):
  File "/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/mdt/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/mdt/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/mdt/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/mdt/lib/python3.9/site-packages/torch/utils/data/_utils/pin_memory.py", line 54, in _pin_memory_loop
    do_one_step()
  File "/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/mdt/lib/python3.9/site-packages/torch/utils/data/_

KeyboardInterrupt: 

In [6]:
global_idx = 0

task_skip_idx = 3
fps = 30

# task_skip_idx = 35
# fps = 5

print("Total tasks:", len(dataset.tcl_dataset.tasks))

for task_idx, task in enumerate(dataset.tcl_dataset.tasks):
    task_length = dataset.tcl_dataset.task_lengths[task_idx]
    images_primary, images_gripper = [], []
    images_cat = []
    actions = []

    if task_idx < task_skip_idx:
        global_idx += task_length
        continue

    for frame_idx in tqdm(range(task_length)):
        frame_data = dataset.tcl_dataset[global_idx]
        global_idx += 1
        print(frame_data.keys())
        images_primary.append(frame_data['rgb_obs']['rgb_static'])
        images_gripper.append(frame_data['rgb_obs']['rgb_gripper'])
        images_cat.append(concatenate_rgb_images(frame_data['primary_rgb'], frame_data['gripper_rgb'], vertical=True, resize_ratio=1))
        actions.append(frame_data['rel_actions'])

    all_vis = []
    actions_vis, fig, ax = plot_action_wrt_time(np.array(actions))
    for frame_idx in range(task_length):
        all_vis.append(concatenate_rgb_images(images_cat[frame_idx], actions_vis[frame_idx],
                                              vertical=False, resize_ratio=1))

    mediapy.show_video(all_vis, fps=fps)

    break

Total tasks: 16


  0%|                                                                                                                                 | 0/590 [00:00<?, ?it/s]

dict_keys(['rel_actions', 'primary_rgb', 'gripper_rgb', 'robot_obs', 'language_text'])


KeyError: 'rgb_obs'

In [7]:
batch_data = dataset[3]

obs_image = []
obs_gripper = []
obs_joint_state = []
action = []

obs_image_data = batch_data['rgb_obs']['rgb_static']  # (T,C,H,W), in [-1,1]
obs_gripper_data = batch_data['rgb_obs']['rgb_gripper']
obs_image_data = (obs_image_data.permute(0, 2, 3, 1).numpy() * 127.5 + 127.5).astype(np.uint8)
obs_gripper_data = (obs_gripper_data.permute(0, 2, 3, 1).numpy() * 127.5 + 127.5).astype(np.uint8)
for obs_image_frame in obs_image_data:
    obs_image.append(obs_image_frame)
for obs_gripper_frame in obs_gripper_data:
    obs_gripper.append(obs_gripper_frame)
mediapy.show_video(obs_image, fps=5)
mediapy.show_video(obs_gripper, fps=5)

action_data = batch_data['actions']  # (T,7)
for action_frame in action_data:
    action.append(action_frame)  # each is (7)
actions_vis, fig, ax = plot_action_wrt_time(np.array(action))
mediapy.show_video(actions_vis, fps=5)



Plotting action dynamic figures...


This browser does not support the video tag.


In [18]:
''' Transform Test '''
## Add transform ##
from torchvision.transforms import transforms
from PIL import Image

trans_crop = transforms.RandomResizedCrop(
    size=(224, 224),
    scale=(0.8, 1.0),  # 从50%到100%都可能 crop
    ratio=(0.9, 1.1)
)

global_idx = 0

task_skip_idx = 3
fps = 30

# task_skip_idx = 35
# fps = 5

print("Total tasks:", len(dataset.tcl_dataset.tasks))

for task_idx, task in enumerate(dataset.tcl_dataset.tasks):
    task_length = dataset.tcl_dataset.task_lengths[task_idx]
    images_primary, images_gripper = [], []
    images_cat = []
    images_aug = []
    actions = []

    # if task_idx < task_skip_idx:
    #     global_idx += task_length
    #     continue

    for frame_idx in tqdm(range(task_length)):
        frame_data = dataset.tcl_dataset[global_idx]
        global_idx += 1
        print(frame_data.keys())
        images_primary.append(frame_data['primary_rgb'])
        images_gripper.append(frame_data['gripper_rgb'])
        images_cat.append(concatenate_rgb_images(frame_data['primary_rgb'], frame_data['gripper_rgb'], vertical=True, resize_ratio=1))
        actions.append(frame_data['rel_actions'])

        print(frame_data['language_text'])

        mediapy.show_video(images_cat, fps=fps)

        ## Augmented Images ##
        for _ in range(20):
            aug_primary = np.array(trans_crop(Image.fromarray(frame_data['primary_rgb'])))
            aug_gripper = np.array(trans_crop(Image.fromarray(frame_data['gripper_rgb'])))
            print("ori:", frame_data['primary_rgb'].shape, "aug:", aug_primary.shape)
            images_aug.append(concatenate_rgb_images(aug_primary, aug_gripper, vertical=True, resize_ratio=1))
        
        mediapy.show_video(images_aug, fps=5)

        break

    # all_vis = []
    # actions_vis, fig, ax = plot_action_wrt_time(np.array(actions))
    # for frame_idx in range(task_length):
    #     all_vis.append(concatenate_rgb_images(images_cat[frame_idx], actions_vis[frame_idx],
    #                                           vertical=False, resize_ratio=1))

    # mediapy.show_video(all_vis, fps=fps)

    break



# action_data = batch_data['actions']  # (T,7)
# for action_frame in action_data:
#     action.append(action_frame)  # each is (7)
# actions_vis, fig, ax = plot_action_wrt_time(np.array(action))
# mediapy.show_video(actions_vis, fps=5)

Total tasks: 93


  0%|                                                                                                                                | 0/1751 [00:00<?, ?it/s]

dict_keys(['rel_actions', 'primary_rgb', 'gripper_rgb', 'robot_obs', 'language_text'])
use a spoon to scoop one spoonful of coffee beans from the source cup, then pour the beans into the target cup.#


This browser does not support the video tag.


ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)
ori: (240, 320, 3) aug: (224, 224, 3)


This browser does not support the video tag.


  0%|                                                                                                                                | 0/1751 [00:00<?, ?it/s]
